# **Lab Sheet 06 - Fine Tuning an LLM using LoRA/QLoRA**

####**1. Install the required libraries: transformers, datasets, peft, trl, accelerate, and bitsandbytes.**

In [12]:
import pyarrow
import transformers
import datasets
import peft
import trl
import accelerate

print("PyArrow version:", pyarrow.__version__)
print("All required libraries imported successfully!")

PyArrow version: 23.0.1
All required libraries imported successfully!


In [13]:
import torch
import peft
import trl
import importlib.util

print("PyTorch version:", torch.__version__)
print("PEFT version:", peft.__version__)
print("TRL version:", trl.__version__)
print("torchao installed:", importlib.util.find_spec("torchao") is not None)

PyTorch version: 2.14.1+cu130
PEFT version: 0.21.3
TRL version: 1.18.1
torchao installed: False


####**2. Check GPU availability**

In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print("GPU Name:", gpu_name)
    print(f"GPU Memory: {gpu_memory:.2f} GB")
else:
    print("No GPU available.")


CUDA available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


###**What GPU is available?**

Available GPU is GPU NVIDIA Tesla T4 with 15 GB of GPU memory

###**Why is a GPU useful for LLM fine-tuning?**

A GPU is useful for fine-tuning LLM models because its hardware is built to handle massive repetitive matrix and tensor math that can perform mathematical operations in parallel which speedes up training compared to a CPU.


In [15]:
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

####**3. Load the pretrained language model**

In [16]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded successfully!")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


This model has its 0.5 billion parameters considerably smaller than those of other LLMs.

####**4. Create the training dataset**

Creating 35 questions and answers about fictional computer shop called Techzone.

In [17]:
qa_pairs = [
    ("What is the name of the shop?", "The shop is called TechZone."),
    ("Where is TechZone located?", "TechZone is located in Colombo, Sri Lanka."),
    ("What are the opening hours?", "TechZone is open from 9 AM to 7 PM."),
    ("Is TechZone open on Sundays?", "No, TechZone is closed on Sundays."),
    ("What products does TechZone sell?", "TechZone sells laptops, desktops, keyboards, mice, monitors, and accessories."),
    ("Does TechZone sell gaming laptops?", "Yes, TechZone sells gaming laptops."),
    ("Does TechZone offer laptop repairs?", "Yes, TechZone provides laptop repair services."),
    ("Does TechZone provide delivery?", "Yes, TechZone provides islandwide delivery."),
    ("How much is the delivery fee?", "The standard delivery fee is Rs. 500."),
    ("Does TechZone offer free delivery?", "Yes, orders above Rs. 100,000 qualify for free delivery."),
    ("What payment methods are accepted?", "TechZone accepts cash, debit cards, credit cards, and bank transfers."),
    ("Does TechZone offer installment payments?", "Yes, installment payments are available for selected credit cards."),
    ("What is the warranty for laptops?", "Laptops come with a one-year warranty."),
    ("Can customers return products?", "Yes, eligible products can be returned within 7 days."),
    ("Does TechZone sell second-hand laptops?", "No, TechZone sells only brand-new laptops."),
    ("Does TechZone sell keyboards?", "Yes, TechZone sells wired and wireless keyboards."),
    ("Does TechZone sell wireless mice?", "Yes, wireless mice are available."),
    ("Does TechZone sell monitors?", "Yes, TechZone sells monitors in different sizes."),
    ("Does TechZone sell printers?", "Yes, TechZone sells inkjet and laser printers."),
    ("Does TechZone provide printer repairs?", "No, TechZone does not provide printer repair services."),
    ("Does TechZone sell computer speakers?", "Yes, computer speakers are available."),
    ("Does TechZone sell headphones?", "Yes, TechZone sells wired and wireless headphones."),
    ("Does TechZone sell SSDs?", "Yes, TechZone sells SSD storage drives."),
    ("Does TechZone sell RAM?", "Yes, TechZone sells RAM upgrades."),
    ("Can TechZone upgrade laptop RAM?", "Yes, TechZone provides laptop RAM upgrade services."),
    ("Does TechZone install operating systems?", "Yes, TechZone provides operating system installation."),
    ("Does TechZone sell antivirus software?", "Yes, antivirus software is available."),
    ("Does TechZone provide technical support?", "Yes, technical support is available during working hours."),
    ("How can customers contact TechZone?", "Customers can contact TechZone at 0112345678."),
    ("Does TechZone have a website?", "Yes, the fictional shop website is www.techzone.example."),
    ("Does TechZone offer student discounts?", "Yes, students receive a 5 percent discount on selected products."),
    ("Does TechZone offer bulk discounts?", "Yes, bulk purchase discounts are available."),
    ("Can customers order products online?", "Yes, customers can place orders through the TechZone website."),
    ("Does TechZone sell laptop chargers?", "Yes, compatible laptop chargers are available."),
    ("Does TechZone offer custom desktop builds?", "Yes, TechZone provides custom desktop building services.")
]

df = pd.DataFrame(qa_pairs, columns=["question", "answer"])

print("Total Q&A pairs:", len(df))
display(df.head())


Total Q&A pairs: 35


,question,answer
0,What is the name of the shop?,The shop is called TechZone.
1,Where is TechZone located?,"TechZone is located in Colombo, Sri Lanka."
2,What are the opening hours?,TechZone is open from 9 AM to 7 PM.
3,Is TechZone open on Sundays?,"No, TechZone is closed on Sundays."
4,What products does TechZone sell?,"TechZone sells laptops, desktops, keyboards, m..."


In [18]:
def format_example(row):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful customer service assistant for TechZone computer shop."
        },
        {
            "role": "user",
            "content": row["question"]
        },
        {
            "role": "assistant",
            "content": row["answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_example)

print(dataset[0]["text"])


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful customer service assistant for TechZone computer shop.<|im_end|>
<|im_start|>user
What is the name of the shop?<|im_end|>
<|im_start|>assistant
The shop is called TechZone.<|im_end|>



Before training we need to see how the original model answers questions.

In [19]:
def generate_answer(model, question):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful customer service assistant for TechZone computer shop."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return answer.strip()


Test questions

In [25]:
original_answers = []

def generate_answer(model, question):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful customer service assistant for TechZone computer shop."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Convert messages to text
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text to PyTorch tensors
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Move inputs to the model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get the input length from the actual tensor
    input_length = inputs["input_ids"].shape[1]

    # Extract only newly generated tokens
    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


In [26]:
question = "Where is TechZone located?"

answer = generate_answer(model, question)

print("Question:", question)
print("Original Model:", answer)


Question: Where is TechZone located?
Original Model: TechZone is located in the heart of downtown, near the intersection of Main Street and University Avenue. It's situated on a large commercial block with multiple floors offering various services including repair shops, electronics stores, and other tech-related businesses. The location provides easy access to both residential areas and business districts.


In [27]:
test_questions = [
    "Where is TechZone located?",
    "What are TechZone opening hours?",
    "Does TechZone offer student discounts?",
    "Can I get my laptop repaired at TechZone?",
    "Is delivery free for an order of Rs. 120,000?",
    "Can I buy a used laptop at TechZone?",
    "What is the laptop warranty period?",
    "Can TechZone build a desktop computer for me?"
]

original_answers = []

for question in test_questions:

    answer = generate_answer(model, question)

    original_answers.append(answer)

    print("Question:", question)
    print("Original Model:", answer)
    print("-" * 60)


Question: Where is TechZone located?
Original Model: TechZone is located in the heart of downtown, near the intersection of Main Street and University Avenue. It's situated on a large commercial block with multiple floors offering various services including repair shops, electronics stores, and other tech-related businesses. The location provides easy access to both residential areas and business districts.
------------------------------------------------------------
Question: What are TechZone opening hours?
Original Model: I'm sorry, but I don't have access to real-time information about the current opening hours of TechZone. It's best to check their official website or call them directly to get the most up-to-date and accurate information on their business hours.
------------------------------------------------------------
Question: Does TechZone offer student discounts?
Original Model: I'm sorry, but I don't have access to specific information about any particular tech shop like Te

#####**5. Configure LoRA using PEFT**

It fine-tunes a small number of additional parameters rather than updating all the parameters in the original model.

In [29]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

####**6. Fine-tune the model using SFTTrainer**

In [30]:
training_args = SFTConfig(
    output_dir="./techzone_lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

In [31]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
    processing_class=tokenizer
)

print("SFTTrainer initialized successfully!")


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

SFTTrainer initialized successfully!


In [32]:
trainer.train()


Step,Training Loss
5,4.984122
10,3.899448
15,3.121820
20,2.748052
25,2.545573


TrainOutput(global_step=25, training_loss=3.459803276062012, metrics={'train_runtime': 30.8511, 'train_samples_per_second': 5.672, 'train_steps_per_second': 0.81, 'total_flos': 17966385609216.0, 'train_loss': 3.459803276062012, 'epoch': 5.0})

The training loss will ideally decrease as training progresses. It indicates that the model is learning to predict the training examples more accurately, but it doesn't mean the model will perform better

**Save the fine-tuned model**


In [34]:
trainer.save_model("./techzone_finetuned")
tokenizer.save_pretrained("./techzone_finetuned")

('./techzone_finetuned/tokenizer_config.json',
 './techzone_finetuned/chat_template.jinja',
 './techzone_finetuned/tokenizer.json')

####**7. Test the fine-tuned model**

In [35]:
fine_tuned_model = trainer.model

fine_tuned_answers = []

for question in test_questions:
    answer = generate_answer(fine_tuned_model, question)
    fine_tuned_answers.append(answer)

    print("Question:", question)
    print("Fine-Tuned Model:", answer)
    print("-" * 60)


Question: Where is TechZone located?
Fine-Tuned Model: TechZone is located in the heart of downtown, near the corner of 12th Street and 5th Avenue.
------------------------------------------------------------
Question: What are TechZone opening hours?
Fine-Tuned Model: TechZone is open from 8:00 AM to 12:00 AM and from 5:00 PM to 9:00 PM Monday through Friday, and on weekends from 8:00 AM to 11:00 PM.
------------------------------------------------------------
Question: Does TechZone offer student discounts?
Fine-Tuned Model: Yes, we do have student discounts available to our customers. Please let us know if you would like to apply for one.
------------------------------------------------------------
Question: Can I get my laptop repaired at TechZone?
Fine-Tuned Model: Yes, we can help you with repairs and upgrades to your laptop. Please let us know the issue you're experiencing so that we can assist you better.
------------------------------------------------------------
Question: Is

####**Compare original and fine-tuned results**

In [36]:
comparison_df = pd.DataFrame({
    "Question": test_questions,
    "Original Model": original_answers,
    "Fine-Tuned Model": fine_tuned_answers
})

display(comparison_df)


,Question,Original Model,Fine-Tuned Model
0,Where is TechZone located?,"TechZone is located in the heart of downtown, ...","TechZone is located in the heart of downtown, ..."
1,What are TechZone opening hours?,"I'm sorry, but I don't have access to real-tim...",TechZone is open from 8:00 AM to 12:00 AM and ...
2,Does TechZone offer student discounts?,"I'm sorry, but I don't have access to specific...","Yes, we do have student discounts available to..."
3,Can I get my laptop repaired at TechZone?,"I'm sorry, but as an AI language model, I don'...","Yes, we can help you with repairs and upgrades..."
4,"Is delivery free for an order of Rs. 120,000?","Yes, the delivery fee is typically included in...","Yes, the delivery is free if you place your or..."
5,Can I buy a used laptop at TechZone?,"Yes, you can definitely purchase a used laptop...","Yes, we do sell used laptops at our store. Ple..."
6,What is the laptop warranty period?,The laptop warranty period can vary depending ...,The warranty period of our laptops varies depe...
7,Can TechZone build a desktop computer for me?,"Yes, we can definitely help you with that! We ...","Yes, we can help you with that! Please let us ..."


####Adding more questions

In [37]:
expected_answers = [
    "TechZone is located in Colombo, Sri Lanka.",
    "TechZone is open from 9 AM to 7 PM.",
    "Students receive a 5 percent discount on selected products.",
    "Yes, TechZone provides laptop repair services.",
    "Yes, orders above Rs. 100,000 qualify for free delivery.",
    "No, TechZone sells only brand-new laptops.",
    "Laptops come with a one-year warranty.",
    "Yes, TechZone provides custom desktop building services."
]

comparison_df["Expected Answer"] = expected_answers

display(comparison_df)


,Question,Original Model,Fine-Tuned Model,Expected Answer
0,Where is TechZone located?,"TechZone is located in the heart of downtown, ...","TechZone is located in the heart of downtown, ...","TechZone is located in Colombo, Sri Lanka."
1,What are TechZone opening hours?,"I'm sorry, but I don't have access to real-tim...",TechZone is open from 8:00 AM to 12:00 AM and ...,TechZone is open from 9 AM to 7 PM.
2,Does TechZone offer student discounts?,"I'm sorry, but I don't have access to specific...","Yes, we do have student discounts available to...",Students receive a 5 percent discount on selec...
3,Can I get my laptop repaired at TechZone?,"I'm sorry, but as an AI language model, I don'...","Yes, we can help you with repairs and upgrades...","Yes, TechZone provides laptop repair services."
4,"Is delivery free for an order of Rs. 120,000?","Yes, the delivery fee is typically included in...","Yes, the delivery is free if you place your or...","Yes, orders above Rs. 100,000 qualify for free..."
5,Can I buy a used laptop at TechZone?,"Yes, you can definitely purchase a used laptop...","Yes, we do sell used laptops at our store. Ple...","No, TechZone sells only brand-new laptops."
6,What is the laptop warranty period?,The laptop warranty period can vary depending ...,The warranty period of our laptops varies depe...,Laptops come with a one-year warranty.
7,Can TechZone build a desktop computer for me?,"Yes, we can definitely help you with that! We ...","Yes, we can help you with that! Please let us ...","Yes, TechZone provides custom desktop building..."


This lab demonstrated the process of fine-tuning a pretrained language model using LoRA and supervised fine-tuning. It highlighted the importance of dataset quality, training configuration, and comparing model responses to evaluate the effectiveness of fine-tuning.

- In this experiment, the Qwen2.5-0.5B-Instruct language model was fine-tuned using LoRA on a custom dataset containing 35 question-and-answer pairs about a computer shop.

- The original and fine-tuned models were tested using the same questions to compare their responses.

- The training loss was monitored to observe the model's learning progress.

- The fine-tuned model was expected to provide more accurate, domain-specific answers based on the training dataset.

- LoRA allows language models to be fine-tuned efficiently without updating all their original parameters, reducing memory and computational requirements.

- The experiment also demonstrated that fine-tuning can help a model learn domain-specific information. However, a decrease in training loss does not necessarily guarantee better performance on unseen questions.

The main challenges involved configuring the Google Colab environment, managing GPU memory, selecting appropriate LoRA parameters, and preparing the dataset in the required format.